# Exercise 1: Feature, bundle and sentence accuracy

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Evaluation and Benchmarks · version 0.1.0*

> Compare feature, exact-bundle and observed sentence-exact accuracy on the same morphological predictions, for Kazakh and Turkish UD test sets. Explain the independence assumption behind a product estimate and where it fails.

**Setup.** `pip install "stanza==1.11.0" sacrebleu torch transformers sentencepiece llama-cpp-python fasttext-wheel pyarrow requests "turkicnlp[hfst]==0.5.0"`; each notebook uses only part of this. Exercises 1 and 2 tag the **UD 2.17** Kazakh-KTB and Turkish-IMST test files with Stanza 1.11.0 over the gold words (models from `STANZA_RESOURCES_DIR`, files cached in `UD217_DIR` and checked against their SHA-256 hashes). Exercise 3 uses the X-WMT English–Turkish test set (TIL release, commit `7d988cc`), TurkicNLP's Apertium analyser and generator for Turkish and NLLB-200 distilled 600M (`NLLB_PATH`). Exercises 4 and 5 use Qwen2.5-1.5B-Instruct as an 8-bit GGUF file run with llama.cpp on the CPU (`QWEN_GGUF`, its tokeniser from `QWEN_TOKENIZER`) and GlotLID v3 (`GLOTLID_PATH`). Exercise 4 needs the Kazakh TUMLU-mini files (`TUMLU_DIR`, revision `3523de7`), checked against the hashes of the book's audit. Model outputs are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`).

Stanza 1.11 tags the gold words of the UD 2.17 Kazakh-KTB and Turkish-IMST test files. Feature recall counts gold feature values that the prediction reproduces; precision counts predicted values that are in the gold bundle; an exact bundle must match completely, with absent and extra features both counting as errors.

In [ ]:
import collections, hashlib, json, math, os, random, re, statistics, unicodedata
from pathlib import Path
import requests

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
UD = "https://raw.githubusercontent.com/UniversalDependencies/{repo}/r2.17/{file}"
SHA256 = {   # UD 2.17 release files
    "UD_Turkish-IMST/tr_imst-ud-train.conllu": "8d8b8b581dab9362e6f2e4328aef77b9fabec1d189a199e320ba136710c7bd83",
    "UD_Turkish-IMST/tr_imst-ud-test.conllu": "fc9269bd598489b942a025b678779557266ce0c054a843e47c7db386d76d6459",
    "UD_Kazakh-KTB/kk_ktb-ud-train.conllu": "6297597914a3f319db573573f927a6577c702f5fa00dcd653162aab46026aeaa",
    "UD_Kazakh-KTB/kk_ktb-ud-test.conllu": "1100846ee50c2f970562a21dca8415a381c470ea06bdb8496ae213d59c3a3c4e",
}
UD_DIR = Path(os.environ.get("UD217_DIR", "~/.cache/ud217")).expanduser()
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def ud_path(key):
    path = UD_DIR / key
    if not path.exists():
        repo, file = key.split("/")
        r = requests.get(UD.format(repo=repo, file=file), timeout=120); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    if sha256(path) != SHA256[key]:
        raise ValueError(f"{key} differs from the UD 2.17 release file")
    return path

Word = collections.namedtuple("Word", "id form lemma upos feats")
def read(key):
    """Sentences (lists of Word) of a UD file; multiword-token ranges and empty nodes are skipped."""
    sents, words = [], []
    for line in ud_path(key).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words: sents.append(words)
            words = []
        elif not line.startswith("#"):
            c = line.split("\t")
            if c[0].isdigit():
                words.append(Word(int(c[0]), c[1], c[2], c[3], c[5]))
    return sents

def feats(s):
    return dict(f.split("=", 1) for f in (s or "_").split("|") if "=" in f)

def tag(lang, key):
    """Stanza 1.11 POS/feature/lemma predictions for the gold words of a UD file (cached)."""
    gold = read(key)
    path = CACHE / f"ch15-stanza-{lang}-{SHA256[key][:12]}.json"
    if not path.exists():
        import logging, stanza
        logging.getLogger("stanza").setLevel(logging.ERROR)
        nlp = stanza.Pipeline(lang, processors="tokenize,pos,lemma", tokenize_pretokenized=True, verbose=False,
                              model_dir=os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser())),
                              download_method=None)
        doc = nlp([[w.form for w in s] for s in gold])
        pred = [[[w.lemma or "_", w.upos or "_", w.feats or "_"] for w in s.words] for s in doc.sentences]
        path.write_text(json.dumps(pred, ensure_ascii=False))
    pred = json.loads(path.read_text())
    assert [len(s) for s in pred] == [len(s) for s in gold]
    return gold, [[Word(g.id, g.form, *p) for g, p in zip(gs, ps)] for gs, ps in zip(gold, pred)]

def wilson(k, n, z=1.959963984540054):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

# ---- Qwen2.5-1.5B-Instruct (llama.cpp, q8_0) and GlotLID, for Exercises 4 and 5 ---------------
QWEN_GGUF = os.environ.get("QWEN_GGUF", str(CACHE / "models/qwen2.5-1.5b-instruct-q8_0.gguf"))
_LLM = {}
def llm():
    if "m" not in _LLM:
        from llama_cpp import Llama
        _LLM["m"] = Llama(model_path=QWEN_GGUF, n_ctx=4096, n_threads=THREADS, seed=0, verbose=False)
    return _LLM["m"]

REPLIES = CACHE / "ch15-replies.jsonl"
_CACHED = {}
if REPLIES.exists():
    for line in REPLIES.read_text(encoding="utf-8").splitlines():
        r = json.loads(line); _CACHED[r["key"]] = r["value"]
def cached(key_obj, compute):
    key = hashlib.sha1(json.dumps(key_obj, ensure_ascii=False).encode()).hexdigest()
    if key not in _CACHED:
        _CACHED[key] = compute()
        with REPLIES.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"key": key, "input": key_obj, "value": _CACHED[key]}, ensure_ascii=False) + "\n")
    return _CACHED[key]

def chat(messages, max_tokens=120):
    """Greedy reply (cached)."""
    return cached(["chat", messages, max_tokens], lambda: llm().create_chat_completion(
        messages=messages, temperature=0.0, max_tokens=max_tokens)["choices"][0]["message"]["content"].strip())

_LID = {}
def lid(text):
    if "m" not in _LID:
        import fasttext
        path = os.environ.get("GLOTLID_PATH")
        if not path:
            from huggingface_hub import hf_hub_download
            path = hf_hub_download("cis-lmu/glotlid", "model_v3.bin")
        _LID["m"] = fasttext.load_model(path)
    labels, probs = _LID["m"].predict([text.replace("\n", " ")], k=1)
    return labels[0][0].replace("__label__", ""), float(probs[0][0])

print("UD files:", ", ".join(k.split("/")[1] for k in SHA256))

In [ ]:
def measures(gold, pred):
    n = fr = fp = fg = exact = lem = both = upos = 0
    sent_exact, expected = 0, []
    for gs, ps in zip(gold, pred):
        ok_all = True
        for g, p in zip(gs, ps):
            G, P = feats(g.feats), feats(p.feats)
            hit = sum(P.get(k) == v for k, v in G.items())
            fr += hit; fg += len(G); fp += len(P)
            e = G == P; exact += e; ok_all &= e
            lem += p.lemma == g.lemma; both += e and p.lemma == g.lemma; upos += p.upos == g.upos; n += 1
        sent_exact += ok_all
    acc = exact / n
    for gs in gold:
        expected.append(acc ** len(gs))           # independence: every word correct with probability acc
    prec, rec = fr / fp if fp else float("nan"), fr / fg if fg else float("nan")
    return dict(words=n, sents=len(gold), upos=upos / n, feat_recall=rec, feat_precision=prec, bundle=acc,
                lemma=lem / n, lemma_bundle=both / n, sent_exact=sent_exact / len(gold), independence=statistics.mean(expected))
print(f"{'treebank':<14}{'words':>7}{'sents':>6}{'UPOS':>7}{'feat R':>8}{'feat P':>8}{'bundle':>8}{'lemma':>7}"
      f"{'lem+bun':>8}{'sent exact':>11}{'if indep.':>10}")
for lang, key in (("kk", "UD_Kazakh-KTB/kk_ktb-ud-test.conllu"), ("tr", "UD_Turkish-IMST/tr_imst-ud-test.conllu")):
    gold, pred = tag(lang, key)
    m = measures(gold, pred)
    print(f"{key.split('/')[0][3:]:<14}{m['words']:>7,}{m['sents']:>6}{m['upos']:7.1%}{m['feat_recall']:8.1%}{m['feat_precision']:8.1%}"
          f"{m['bundle']:8.1%}{m['lemma']:7.1%}{m['lemma_bundle']:8.1%}{m['sent_exact']:11.1%}{m['independence']:10.1%}")
    # Observed vs. independence estimate by sentence length.
    rows = collections.defaultdict(lambda: [0, 0])
    for gs, ps in zip(gold, pred):
        b = "1-5" if len(gs) <= 5 else "6-10" if len(gs) <= 10 else "11-20" if len(gs) <= 20 else "21+"
        rows[b][0] += 1; rows[b][1] += all(feats(g.feats) == feats(p.feats) for g, p in zip(gs, ps))
    print("   sentence-exact by length:", {b: f"{k}/{n} = {k / n:.0%} (indep. {m['bundle'] ** {'1-5': 3, '6-10': 8, '11-20': 15, '21+': 26}[b]:.0%})"
                                          for b, (n, k) in sorted(rows.items(), key=lambda x: int(x[0].split('-')[0].rstrip('+')))})
    errs = collections.Counter((k, feats(g.feats).get(k), feats(p.feats).get(k)) for gs, ps in zip(gold, pred) for g, p in zip(gs, ps)
                               for k in set(feats(g.feats)) | set(feats(p.feats)) if feats(g.feats).get(k) != feats(p.feats).get(k))
    print("   most frequent feature errors (feature, gold, predicted):", errs.most_common(5))
print("\nthe chapter's example: gold Case=Acc|Number=Sing, predicted Case=Nom|Number=Sing -> feature accuracy",
      sum(feats("Case=Nom|Number=Sing").get(k) == v for k, v in feats("Case=Acc|Number=Sing").items()) / 2,
      "| exact bundle", feats("Case=Nom|Number=Sing") == feats("Case=Acc|Number=Sing"))

## Worked solution

Output of `experiments/ch15_exercises/run_ch15_exercises.py`:

```
treebank        words sents   UPOS  feat R  feat P  bundle  lemma lem+bun sent exact if indep.
Kazakh-KTB     10,007  1047  98.7%   98.8%   98.9%   98.2%  98.7%   97.4%      89.0%     84.0%
   sentence-exact by length: {'1-5': '173/198 = 87% (indep. 95%)', '6-10': '469/523 = 90% (indep. 86%)', '11-20': '253/287 = 88% (indep. 76%)', '21+': '37/39 = 95% (indep. 62%)'}
   most frequent feature errors (feature, gold, predicted): [(('Case', None, 'Nom'), 23), (('Case', 'Nom', None), 17), (('Number', 'Plur', 'Sing'), 11), (('Polite', 'Form', None), 8), (('Voice', 'Rcp', None), 8)]
Turkish-IMST   10,032  1100  95.2%   95.0%   95.6%   91.3%  96.0%   88.9%      50.7%     50.3%
   sentence-exact by length: {'1-5': '275/394 = 70% (indep. 76%)', '6-10': '235/429 = 55% (indep. 48%)', '11-20': '43/187 = 23% (indep. 25%)', '21+': '5/90 = 6% (indep. 9%)'}
   most frequent feature errors (feature, gold, predicted): [(('Number', 'Sing', None), 137), (('Person', '3', None), 136), (('Person', None, '3'), 131), (('Number', None, 'Sing'), 127), (('Case', 'Nom', None), 117)]

the chapter's example: gold Case=Acc|Number=Sing, predicted Case=Nom|Number=Sing -> feature accuracy 0.5 | exact bundle False
```

**Feature, bundle and sentence scores describe different units.** On Turkish IMST, feature recall is 95.0% and precision is 95.6%; exact-bundle accuracy is 91.3%. Lemma-plus-bundle accuracy is 88.9%. All feature bundles match in 50.7% of sentences. This last score does not require correct lemmas or UPOS. The denominators include punctuation and use gold word boundaries.

**The product assumes independent, equally likely errors.** Raising the overall bundle accuracy to each sentence's actual length and averaging gives 50.3%. Its closeness to the observed 50.7% does not validate those assumptions. The printed length-bin estimates instead use fixed lengths of 3, 8, 15 and 26 words. They are rough illustrations, not the mean product prediction within each bin.

Kazakh's observed sentence rate is 89.0%, against an 84.0% product estimate. This difference alone does not identify its cause. The Kazakh model was trained on the KTB test file (Syntax and Universal Dependencies), so these scores illustrate the arithmetic, not held-out performance.

## Your turn — extension

1. Count absent and extra features separately and report both precision and recall per feature (`Case`, `Number`, `Person`, `Poss`).
2. Repeat on Turkish BOUN with the IMST model and compare the gap between feature and bundle accuracy.